In [3]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('/content/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))

Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [4]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## **Task 1 — Histogram: price by room type (overlapping distributions)**

In [5]:
import plotly.express as px

# Filter required room types
filtered = df_cap[
    df_cap['room_type'].isin([
        'Entire home/apt',
        'Private room'
    ])
]

# Create histogram
fig = px.histogram(
    filtered,
    x='price',
    color='room_type',
    barmode='overlay',
    opacity=0.6,
    title='Entire homes are generally more expensive than private rooms'
)

# Calculate medians
median_entire = filtered[
    filtered['room_type'] == 'Entire home/apt'
]['price'].median()

median_private = filtered[
    filtered['room_type'] == 'Private room'
]['price'].median()

# Add median lines
fig.add_vline(
    x=median_entire,
    line_dash='dash',
    annotation_text='Entire Home Median'
)

fig.add_vline(
    x=median_private,
    line_dash='dot',
    annotation_text='Private Room Median'
)

fig.show()

## Task 2 — Box plot: listing activity by **borough** **bold text** **bold text** **bold text**

In [10]:
# Median review values
median_reviews = (
    df_cap.groupby('neighbourhood')['reviews_per_month']
    .median()
    .sort_values(ascending=False)
)

# Top 2 neighbourhoods
top2 = median_reviews.head(2).index.tolist()

# Create clean copy
df_cap = df_cap.copy()

# Highlight top 2
df_cap['highlight'] = df_cap['neighbourhood'].apply(
    lambda x: x if x in top2 else 'Other Areas'
)

# Color mapping
color_map = {
    top2[0]: '#1f77b4',
    top2[1]: '#ff7f0e',
    'Other Areas': 'lightgrey'
}

# Create box plot
fig = px.box(
    df_cap,
    x='reviews_per_month',
    y='neighbourhood',
    color='highlight',
    orientation='h',
    points='outliers',
    category_orders={
        'neighbourhood': median_reviews.index
    },
    color_discrete_map=color_map,
    title='Top neighbourhoods show higher review activity than others'
)

fig.show()